# Threat Intelligence Graph Analytics

Connect synthetic indicators, infrastructure, techniques, campaigns, and enterprise assets into an evidence graph.

**Safety and scope:** This project uses synthetic, non-sensitive telemetry for defensive analytics. It does not perform exploitation or execute malicious content.

## Goal

Implement PageRank and bounded evidence-path search to prioritize connected entities and exposed assets.


## Setup

The notebook is deterministic, runs offline, and implements the core analytical method directly with NumPy and Pandas so the modeling logic remains inspectable.


In [1]:
from collections import defaultdict, deque

import numpy as np
import pandas as pd

pd.set_option("display.width", 130)

def build_adjacency(edges):
    adjacency = defaultdict(set)
    for source, target, relation in edges:
        adjacency[source].add(target)
        adjacency[target].add(source)
    return adjacency

def pagerank(nodes, adjacency, damping=0.85, iterations=80):
    scores = {node: 1.0 / len(nodes) for node in nodes}
    for _ in range(iterations):
        updated = {node: (1 - damping) / len(nodes) for node in nodes}
        for node in nodes:
            neighbors = adjacency[node]
            if neighbors:
                share = damping * scores[node] / len(neighbors)
                for neighbor in neighbors:
                    updated[neighbor] += share
        scores = updated
    return scores

def shortest_path(adjacency, start, goal, max_depth=5):
    queue = deque([(start, [start])])
    visited = {start}
    while queue:
        node, path = queue.popleft()
        if node == goal:
            return path
        if len(path) - 1 >= max_depth:
            continue
        for neighbor in sorted(adjacency[node]):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))
    return None


## Steps

### 1. Build a synthetic CTI graph


In [2]:
node_types = {
    "indicator:alpha.test": "indicator",
    "indicator:203.0.113.50": "indicator",
    "indicator:hash-demo-01": "indicator",
    "infra:edge-relay": "infrastructure",
    "infra:mail-gateway": "infrastructure",
    "malware:sample-a": "malware",
    "campaign:aurora-demo": "campaign",
    "actor:group-demo": "threat_actor",
    "technique:T1566": "technique",
    "technique:T1059": "technique",
    "asset:finance-laptop": "asset",
    "asset:identity-server": "asset",
    "asset:web-server": "asset",
}
edges = [
    ("indicator:alpha.test", "infra:edge-relay", "resolves_to"),
    ("indicator:203.0.113.50", "infra:edge-relay", "hosts"),
    ("indicator:hash-demo-01", "malware:sample-a", "hash_of"),
    ("infra:edge-relay", "campaign:aurora-demo", "used_by"),
    ("infra:mail-gateway", "campaign:aurora-demo", "used_by"),
    ("malware:sample-a", "campaign:aurora-demo", "associated_with"),
    ("campaign:aurora-demo", "actor:group-demo", "attributed_to"),
    ("campaign:aurora-demo", "technique:T1566", "uses"),
    ("malware:sample-a", "technique:T1059", "uses"),
    ("asset:finance-laptop", "indicator:alpha.test", "observed"),
    ("asset:web-server", "indicator:203.0.113.50", "observed"),
    ("asset:identity-server", "infra:mail-gateway", "communicated_with"),
]
adjacency = build_adjacency(edges)
print("Nodes:", len(node_types), "Edges:", len(edges))
print(pd.DataFrame(edges, columns=["source", "target", "relation"]).to_string(index=False))


Nodes: 13 Edges: 12
                source                 target          relation
  indicator:alpha.test       infra:edge-relay       resolves_to
indicator:203.0.113.50       infra:edge-relay             hosts
indicator:hash-demo-01       malware:sample-a           hash_of
      infra:edge-relay   campaign:aurora-demo           used_by
    infra:mail-gateway   campaign:aurora-demo           used_by
      malware:sample-a   campaign:aurora-demo   associated_with
  campaign:aurora-demo       actor:group-demo     attributed_to
  campaign:aurora-demo        technique:T1566              uses
      malware:sample-a        technique:T1059              uses
  asset:finance-laptop   indicator:alpha.test          observed
      asset:web-server indicator:203.0.113.50          observed
 asset:identity-server     infra:mail-gateway communicated_with


### 2. Rank entities and find evidence paths


In [3]:
centrality = pagerank(list(node_types), adjacency)
centrality_table = pd.DataFrame({
    "entity": list(centrality),
    "type": [node_types[node] for node in centrality],
    "pagerank": list(centrality.values()),
    "degree": [len(adjacency[node]) for node in centrality],
}).sort_values("pagerank", ascending=False)

indicators = [node for node, node_type in node_types.items() if node_type == "indicator"]
assets = [node for node, node_type in node_types.items() if node_type == "asset"]
evidence_paths = []
for indicator in indicators:
    for asset in assets:
        path = shortest_path(adjacency, indicator, asset, max_depth=4)
        if path:
            evidence_paths.append({
                "indicator": indicator,
                "asset": asset,
                "hops": len(path) - 1,
                "path": " -> ".join(path),
            })
evidence_table = pd.DataFrame(evidence_paths).sort_values(["hops", "asset", "indicator"])

print("Highest-centrality entities:")
print(centrality_table.head(8).round(4).to_string(index=False))
print("\nBounded indicator-to-asset evidence paths:")
print(evidence_table.head(12).to_string(index=False))


Highest-centrality entities:
                entity           type  pagerank  degree
  campaign:aurora-demo       campaign    0.1884       5
      malware:sample-a        malware    0.1219       3
      infra:edge-relay infrastructure    0.1155       3
  indicator:alpha.test      indicator    0.0847       2
indicator:203.0.113.50      indicator    0.0847       2
    infra:mail-gateway infrastructure    0.0836       2
  asset:finance-laptop          asset    0.0475       1
      asset:web-server          asset    0.0475       1

Bounded indicator-to-asset evidence paths:
             indicator                 asset  hops                                                                                                              path
  indicator:alpha.test  asset:finance-laptop     1                                                                      indicator:alpha.test -> asset:finance-laptop
indicator:203.0.113.50      asset:web-server     1                                           

## Checks


In [4]:
assert abs(sum(centrality.values()) - 1.0) < 1e-6
assert len(evidence_table) >= 3
assert evidence_table["hops"].max() <= 4
assert "campaign:aurora-demo" in set(centrality_table.head(5)["entity"])
print("Checks passed: normalized PageRank, bounded paths, and a central campaign entity.")


Checks passed: normalized PageRank, bounded paths, and a central campaign entity.


## Next Steps

        - Add timestamps, confidence, provenance, and marking constraints to every relationship.
- Score paths using source reliability and enterprise-observation recency.
- Persist the graph in a graph database and expose analyst-friendly evidence views.
